# 80. The International Freight Mode Selection Problem (Air vs. Ocean)
## Tier 2: The Classic Heuristic (Python Implementation)

### Key assumptions
- Priority-based scoring system balances multiple decision factors
- Cost-time trade-offs are quantified through priority functions
- Air freight capacity constraints are handled through greedy allocation
- Real-time decision making requires sub-second computation
- Market conditions are relatively stable during decision horizon

### Approach (step-by-step)
1. **Calculate priority scores**: Multi-criteria evaluation for each shipment
2. **Sort by priority**: Rank shipments from highest to lowest priority
3. **Allocate modes**: Assign air freight to highest priority within capacity
4. **Assign remaining**: Allocate ocean freight to remaining shipments
5. **Calculate costs**: Compute total cost including transport, holding, and penalties

### What to look for in the results
- Priority score breakdown showing urgency, value density, and cost factors
- Mode assignments reflecting priority-based decision logic
- Capacity utilization and constraint satisfaction
- Performance comparison with mathematical optimum

### Concrete example (from the source)
The Cost-Time Priority Heuristic evaluates the same three shipments:
- Shipment 1: 500kg microprocessors, value $2M, due in Rotterdam in 14 days
- Shipment 2: 2000kg components, value $500K, due in LA in 21 days  
- Shipment 3: 5000kg servers, value $1M, due in Rotterdam in 35 days

The heuristic computes priority scores based on:
- **Urgency factor**: Time pressure based on due date proximity
- **Value density**: Shipment value per unit weight
- **Cost differential**: Air vs. ocean cost difference

Expected results: Priority scores of ~890 for Shipment 1 (air), ~42 for Shipment 2 (ocean), and ~18 for Shipment 3 (ocean), achieving 98.9% of optimal cost.

In [ ]:
# Import required packages for heuristic implementation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Shipment:
    """Data class for shipment information"""
    id: int
    weight: float  # kg
    value: float   # USD
    destination: str
    due_date: int  # days from now

@dataclass 
class TransportParams:
    """Parameters for transportation costs and times"""
    costs: Dict[str, Dict[str, float]]  # mode -> destination -> cost per kg
    transit_times: Dict[str, Dict[str, int]]  # mode -> destination -> days
    holding_rate: float  # daily holding cost rate as fraction of value
    late_penalty: float  # penalty per day late
    air_capacity: float  # monthly air freight capacity in kg

# Define the concrete example from the problem description
shipments = [
    Shipment(1, 500, 2000000, "rotterdam", 14),   # High-value microprocessors
    Shipment(2, 2000, 500000, "los_angeles", 21),  # Standard components
    Shipment(3, 5000, 1000000, "rotterdam", 35),   # Large servers
]

params = TransportParams(
    costs={
        "ocean": {"rotterdam": 2.0, "los_angeles": 1.8},
        "air": {"rotterdam": 6.2, "los_angeles": 5.5}
    },
    transit_times={
        "ocean": {"rotterdam": 25, "los_angeles": 18},
        "air": {"rotterdam": 3, "los_angeles": 2}
    },
    holding_rate=0.001,  # 0.1% per day
    late_penalty=1000,   # $1000 per day
    air_capacity=8000   # 8000 kg per month
)

print("Shipments:")
for s in shipments:
    print(f"  Shipment {s.id}: {s.weight}kg, ${s.value:,}, to {s.destination}, due in {s.due_date} days")
print(f"\nAir freight capacity: {params.air_capacity:,} kg/month")

In [ ]:
class FreightModeSelector:
    """
    Cost-Time Priority Heuristic for freight mode selection
    Implements the priority-based scoring system from the source material
    """
    
    def __init__(self, params: TransportParams):
        self.params = params
        self.priority_threshold = 50  # Threshold for air vs ocean decision
    
    def calculate_priority_score(self, shipment: Shipment) -> Tuple[float, str]:
        """
        Calculate priority score for a shipment considering multiple factors:
        - Urgency (time pressure)
        - Value density (value per weight)
        - Cost differential between air and ocean
        
        Returns: (priority_score, recommended_mode)
        """
        # Calculate total costs for both modes
        ocean_cost = self._calculate_total_cost(shipment, 'ocean')
        air_cost = self._calculate_total_cost(shipment, 'air')
        
        # Priority factors calculation
        cost_diff = air_cost - ocean_cost
        ocean_arrival = self.params.transit_times['ocean'][shipment.destination]
        time_buffer = shipment.due_date - ocean_arrival
        
        # Urgency: higher if less time buffer (0-10 scale)
        urgency = max(0, 10 - time_buffer / 3)  # Normalize to 0-10 scale
        
        # Value density: value per kg (scaled for priority calculation)
        value_density = shipment.value / shipment.weight
        
        # Combined priority score: favor air if high urgency or value, low cost diff
        # Formula from source: urgency * 100 + value_density * 0.001 - cost_diff * 0.01
        priority = urgency * 100 + value_density * 0.001 - cost_diff * 0.01
        
        # Recommend mode based on priority threshold
        recommended_mode = 'air' if priority > self.priority_threshold else 'ocean'
        
        return priority, recommended_mode
    
    def _calculate_total_cost(self, shipment: Shipment, mode: str) -> float:
        """
        Calculate total cost for a single shipment using a specific transport mode
        Includes transportation cost, holding cost (if early), and late penalty (if late)
        """
        # Transportation cost
        transport_cost = shipment.weight * self.params.costs[mode][shipment.destination]
        
        # Transit time
        transit_time = self.params.transit_times[mode][shipment.destination]
        
        # Holding cost if arrives early
        if transit_time < shipment.due_date:
            early_days = shipment.due_date - transit_time
            holding_cost = early_days * self.params.holding_rate * shipment.value
        else:
            holding_cost = 0
        
        # Late penalty if arrives after due date
        if transit_time > shipment.due_date:
            late_days = transit_time - shipment.due_date
            late_penalty_cost = late_days * self.params.late_penalty
        else:
            late_penalty_cost = 0
        
        return transport_cost + holding_cost + late_penalty_cost
    
    def select_modes_batch(self, shipments: List[Shipment]) -> Dict[int, str]:
        """
        Select transport modes for multiple shipments using priority-based approach
        Implements the capacity-constrained greedy allocation from the source
        
        Returns: Dictionary mapping shipment_id -> mode
        """
        # Calculate priorities for all shipments
        priorities = []
        for shipment in shipments:
            priority, preferred_mode = self.calculate_priority_score(shipment)
            priorities.append({
                'shipment_id': shipment.id,
                'priority': priority,
                'preferred_mode': preferred_mode,
                'weight': shipment.weight
            })
        
        # Sort by priority descending (highest priority first)
        priorities.sort(key=lambda x: x['priority'], reverse=True)
        
        # Assign modes with air capacity constraint
        assignments = {}
        air_used = 0
        
        for item in priorities:
            shipment_id = item['shipment_id']
            preferred_mode = item['preferred_mode']
            weight = item['weight']
            
            # Check air capacity constraint
            if preferred_mode == 'air' and air_used + weight <= self.params.air_capacity:
                assignments[shipment_id] = 'air'
                air_used += weight
            else:
                # Either prefers ocean or air capacity exceeded
                assignments[shipment_id] = 'ocean'
        
        return assignments, priorities

# Initialize the freight mode selector
selector = FreightModeSelector(params)

print("Cost-Time Priority Heuristic initialized")
print(f"Priority threshold: {selector.priority_threshold}")
print(f"Air capacity: {params.air_capacity:,} kg")

In [ ]:
# Calculate priority scores for all shipments
print("Priority Score Analysis:")
print("=" * 80)

priority_details = []
for shipment in shipments:
    priority, recommended_mode = selector.calculate_priority_score(shipment)
    
    # Calculate component factors for detailed analysis
    ocean_cost = selector._calculate_total_cost(shipment, 'ocean')
    air_cost = selector._calculate_total_cost(shipment, 'air')
    cost_diff = air_cost - ocean_cost
    ocean_arrival = params.transit_times['ocean'][shipment.destination]
    time_buffer = shipment.due_date - ocean_arrival
    urgency = max(0, 10 - time_buffer / 3)
    value_density = shipment.value / shipment.weight
    
    priority_details.append({
        'shipment_id': shipment.id,
        'priority': priority,
        'recommended_mode': recommended_mode,
        'urgency': urgency,
        'value_density': value_density,
        'cost_diff': cost_diff,
        'time_buffer': time_buffer
    })
    
    print(f"\nShipment {shipment.id} ({shipment.destination}):")
    print(f"  Priority Score: {priority:.1f}")
    print(f"  Recommended Mode: {recommended_mode.upper()}")
    print(f"  Components:")
    print(f"    - Urgency factor: {urgency:.2f} (time buffer: {time_buffer} days)")
    print(f"    - Value density: ${value_density:,.0f}/kg")
    print(f"    - Cost differential: ${cost_diff:,.0f} (air vs ocean)")
    print(f"  Cost comparison: Ocean ${ocean_cost:,.0f}, Air ${air_cost:,.0f}")

# Create priority analysis dataframe
priority_df = pd.DataFrame(priority_details)
print(f"\nPriority Ranking:")
priority_df_sorted = priority_df.sort_values('priority', ascending=False)
for _, row in priority_df_sorted.iterrows():
    print(f"  {row['shipment_id']}. Priority {row['priority']:.1f} -> {row['recommended_mode'].upper()}")

In [ ]:
# Apply the heuristic to select modes for all shipments
assignments, priority_details = selector.select_modes_batch(shipments)

print("\nHeuristic Mode Assignment:")
print("=" * 80)

total_cost = 0
air_weight_used = 0
assignment_details = []

for shipment in shipments:
    assigned_mode = assignments[shipment.id]
    cost = selector._calculate_total_cost(shipment, assigned_mode)
    transit_time = params.transit_times[assigned_mode][shipment.destination]
    
    if assigned_mode == 'air':
        air_weight_used += shipment.weight
    
    assignment_details.append({
        'shipment_id': shipment.id,
        'assigned_mode': assigned_mode,
        'cost': cost,
        'transit_time': transit_time,
        'weight': shipment.weight
    })
    
    total_cost += cost
    
    print(f"\nShipment {shipment.id}: {assigned_mode.upper()}")
    print(f"  Cost: ${cost:,.0f}")
    print(f"  Transit time: {transit_time} days (due: {shipment.due_date})")
    print(f"  Weight: {shipment.weight:,} kg")

print(f"\nSummary:")
print(f"  Total heuristic cost: ${total_cost:,.0f}")
print(f"  Air capacity used: {air_weight_used:,} kg / {params.air_capacity:,} kg")
print(f"  Capacity utilization: {air_weight_used/params.air_capacity*100:.1f}%")
print(f"  Shipments by air: {sum(1 for m in assignments.values() if m == 'air')}")
print(f"  Shipments by ocean: {sum(1 for m in assignments.values() if m == 'ocean')}")

In [ ]:
# Compare with baseline strategies and mathematical optimum
def compare_strategies(shipments: List[Shipment], params: TransportParams, 
                     heuristic_cost: float) -> Dict:
    """
    Compare heuristic solution with baseline strategies
    """
    strategies = {}
    
    # Strategy 1: All ocean freight
    ocean_cost = sum(selector._calculate_total_cost(s, 'ocean') for s in shipments)
    strategies['all_ocean'] = {
        'cost': ocean_cost,
        'description': 'All shipments via ocean freight'
    }
    
    # Strategy 2: All air freight (check capacity)
    total_air_weight = sum(s.weight for s in shipments)
    if total_air_weight <= params.air_capacity:
        air_cost = sum(selector._calculate_total_cost(s, 'air') for s in shipments)
        strategies['all_air'] = {
            'cost': air_cost,
            'description': 'All shipments via air freight'
        }
    else:
        strategies['all_air'] = {
            'cost': float('inf'),
            'description': f'Infeasible: exceeds air capacity ({total_air_weight:,} > {params.air_capacity:,} kg)'
        }
    
    # Strategy 3: Heuristic solution
    strategies['heuristic'] = {
        'cost': heuristic_cost,
        'description': 'Cost-Time Priority Heuristic solution'
    }
    
    # Strategy 4: Mathematical optimum (from source material)
    optimal_cost = 142600  # From source: optimal solution cost
    strategies['optimal'] = {
        'cost': optimal_cost,
        'description': 'Mathematical programming optimal solution'
    }
    
    return strategies

# Compare strategies
strategies = compare_strategies(shipments, params, total_cost)

print("\nStrategy Comparison:")
print("=" * 80)
for name, strategy in strategies.items():
    if strategy['cost'] == float('inf'):
        print(f"{name.replace('_', ' ').title()}: {strategy['description']}")
    else:
        print(f"{name.replace('_', ' ').title()}: ${strategy['cost']:,.0f}")
        print(f"  {strategy['description']}")
        
        # Calculate performance vs optimal
        if name != 'optimal' and strategies['optimal']['cost'] != float('inf'):
            gap = strategy['cost'] - strategies['optimal']['cost']
            gap_pct = gap / strategies['optimal']['cost'] * 100
            print(f"  Gap vs optimal: ${gap:,.0f} ({gap_pct:+.1f}%)")
    print()

# Calculate heuristic performance metrics
heuristic_vs_optimal = (strategies['heuristic']['cost'] - strategies['optimal']['cost']) / strategies['optimal']['cost'] * 100
heuristic_vs_ocean = (strategies['all_ocean']['cost'] - strategies['heuristic']['cost']) / strategies['all_ocean']['cost'] * 100

print("Heuristic Performance:")
print(f"  Achieves {100 - abs(heuristic_vs_optimal):.1f}% of mathematical optimum")
print(f"  Saves {heuristic_vs_ocean:.1f}% vs all-ocean strategy")
print(f"  Computation time: Sub-second (real-time capable)")

In [ ]:
# Create comprehensive visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Cost-Time Priority Heuristic Analysis', fontsize=16, fontweight='bold')

# 1. Priority score breakdown
shipment_ids = [detail['shipment_id'] for detail in priority_details]
priorities = [detail['priority'] for detail in priority_details]
urgencies = [detail['urgency'] * 100 for detail in priority_details]  # Scale for visibility
value_densities = [detail['value_density'] * 0.001 for detail in priority_details]
cost_diffs = [detail['cost_diff'] * 0.01 for detail in priority_details]
modes = [detail['recommended_mode'].upper() for detail in priority_details]
colors = ['lightcoral' if mode == 'AIR' else 'lightblue' for mode in modes]

x = np.arange(len(shipment_ids))
width = 0.2

ax1.bar(x - width*1.5, priorities, width, label='Total Priority', alpha=0.8, color='purple')
ax1.bar(x - width*0.5, urgencies, width, label='Urgency Component', alpha=0.8, color='red')
ax1.bar(x + width*0.5, value_densities, width, label='Value Density Component', alpha=0.8, color='gold')
ax1.bar(x + width*1.5, cost_diffs, width, label='Cost Diff Component', alpha=0.8, color='darkgreen')

ax1.set_xlabel('Shipment ID')
ax1.set_ylabel('Priority Score')
ax1.set_title('Priority Score Components by Shipment')
ax1.set_xticks(x)
ax1.set_xticklabels([f'S{id}\n({mode})' for id, mode in zip(shipment_ids, modes)])
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Strategy comparison
strategy_names = []
strategy_costs = []
strategy_colors = []

for name, strategy in strategies.items():
    if strategy['cost'] != float('inf'):
        strategy_names.append(name.replace('_', '\n').title())
        strategy_costs.append(strategy['cost'])
        if name == 'optimal':
            strategy_colors.append('green')
        elif name == 'heuristic':
            strategy_colors.append('blue')
        else:
            strategy_colors.append('orange')

bars = ax2.bar(strategy_names, strategy_costs, color=strategy_colors, alpha=0.7)
ax2.set_ylabel('Total Cost ($)')
ax2.set_title('Strategy Cost Comparison')
ax2.tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar, cost in zip(bars, strategy_costs):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
             f'${cost:,.0f}', ha='center', va='bottom', fontweight='bold')

ax2.grid(True, alpha=0.3)

# 3. Mode assignment visualization
mode_counts = {'Air': 0, 'Ocean': 0}
mode_weights = {'Air': 0, 'Ocean': 0}

for shipment in shipments:
    mode = assignments[shipment.id].title()
    mode_counts[mode] += 1
    mode_weights[mode] += shipment.weight

# Create pie chart for shipment count
ax3.pie(mode_counts.values(), labels=mode_counts.keys(), autopct='%1.0f shipments', 
        colors=['lightcoral', 'lightblue'], startangle=90)
ax3.set_title('Heuristic Mode Assignment')

# 4. Priority vs assignment analysis
priority_scores = [detail['priority'] for detail in priority_details]
assigned_modes = [assignments[detail['shipment_id']].upper() for detail in priority_details]
assignment_colors = ['red' if mode == 'AIR' else 'blue' for mode in assigned_modes]

bars = ax4.bar(shipment_ids, priority_scores, color=assignment_colors, alpha=0.7)
ax4.set_xlabel('Shipment ID')
ax4.set_ylabel('Priority Score')
ax4.set_title('Priority Scores vs Final Assignment')
ax4.axhline(y=selector.priority_threshold, color='green', linestyle='--', 
            label=f'Threshold ({selector.priority_threshold})', linewidth=2)
ax4.legend()
ax4.grid(True, alpha=0.3)

# Add assignment labels on bars
for bar, mode in zip(bars, assigned_modes):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
             mode, ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nVisualization Summary:")
print("=" * 80)
print(f"1. Priority Components: Shows how urgency, value density, and cost contribute to scores")
print(f"2. Strategy Comparison: Heuristic vs baseline strategies and optimal solution")
print(f"3. Mode Assignment: {mode_counts['Air']} shipments by air, {mode_counts['Ocean']} by ocean")
print(f"4. Priority Analysis: Priority scores vs threshold and final assignments")

In [ ]:
# Sensitivity analysis on priority threshold
def threshold_sensitivity_analysis(shipments: List[Shipment], params: TransportParams) -> Dict:
    """
    Analyze how different priority thresholds affect solution quality
    """
    thresholds = [30, 40, 50, 60, 70, 80]
    results = {
        'thresholds': thresholds,
        'costs': [],
        'air_shipments': [],
        'air_weights': []
    }
    
    for threshold in thresholds:
        # Create selector with new threshold
        test_selector = FreightModeSelector(params)
        test_selector.priority_threshold = threshold
        
        # Get assignments
        test_assignments, _ = test_selector.select_modes_batch(shipments)
        
        # Calculate total cost
        test_cost = sum(test_selector._calculate_total_cost(s, test_assignments[s.id]) for s in shipments)
        
        # Count air shipments and weight
        air_count = sum(1 for mode in test_assignments.values() if mode == 'air')
        air_weight = sum(s.weight for s in shipments if test_assignments[s.id] == 'air')
        
        results['costs'].append(test_cost)
        results['air_shipments'].append(air_count)
        results['air_weights'].append(air_weight)
    
    return results

# Perform threshold sensitivity analysis
print("Performing Priority Threshold Sensitivity Analysis...")
threshold_analysis = threshold_sensitivity_analysis(shipments, params)

# Create threshold sensitivity plots
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Priority Threshold Sensitivity Analysis', fontsize=16, fontweight='bold')

# Cost vs threshold
ax1.plot(threshold_analysis['thresholds'], threshold_analysis['costs'], 
         'o-', linewidth=2, markersize=8, color='steelblue')
ax1.set_xlabel('Priority Threshold')
ax1.set_ylabel('Total Cost ($)')
ax1.set_title('Impact of Threshold on Total Cost')
ax1.grid(True, alpha=0.3)
ax1.axvline(x=50, color='red', linestyle='--', label='Current Threshold (50)', linewidth=2)
ax1.axhline(y=strategies['optimal']['cost'], color='green', linestyle='--', 
            label=f'Optimal (${strategies["optimal"]["cost"]:,.0f})', linewidth=2)
ax1.legend()

# Air shipments vs threshold
ax2.plot(threshold_analysis['thresholds'], threshold_analysis['air_shipments'], 
         'o-', linewidth=2, markersize=8, color='darkorange')
ax2.set_xlabel('Priority Threshold')
ax2.set_ylabel('Number of Air Shipments')
ax2.set_title('Air Shipment Count vs Threshold')
ax2.grid(True, alpha=0.3)
ax2.axvline(x=50, color='red', linestyle='--', label='Current Threshold (50)', linewidth=2)
ax2.legend()

# Air weight vs threshold
ax3.plot(threshold_analysis['thresholds'], threshold_analysis['air_weights'], 
         'o-', linewidth=2, markersize=8, color='crimson')
ax3.set_xlabel('Priority Threshold')
ax3.set_ylabel('Air Weight (kg)')
ax3.set_title('Air Weight Utilization vs Threshold')
ax3.grid(True, alpha=0.3)
ax3.axvline(x=50, color='red', linestyle='--', label='Current Threshold (50)', linewidth=2)
ax3.axhline(y=params.air_capacity, color='green', linestyle='--', 
            label=f'Capacity ({params.air_capacity:,} kg)', linewidth=2)
ax3.legend()

plt.tight_layout()
plt.show()

print("\nThreshold Sensitivity Results:")
print("=" * 80)
for i, threshold in enumerate(threshold_analysis['thresholds']):
    cost = threshold_analysis['costs'][i]
    air_shipments = threshold_analysis['air_shipments'][i]
    air_weight = threshold_analysis['air_weights'][i]
    gap_pct = (cost - strategies['optimal']['cost']) / strategies['optimal']['cost'] * 100
    
    print(f"Threshold {threshold}:")
    print(f"  Cost: ${cost:,.0f} ({gap_pct:+.1f}% vs optimal)")
    print(f"  Air shipments: {air_shipments}, Weight: {air_weight:,} kg")
    print(f"  Capacity utilization: {air_weight/params.air_capacity*100:.1f}%")

### Why this Tier exists vs earlier Tiers
The Cost-Time Priority Heuristic addresses key limitations of the mathematical approach:
- **Real-time performance**: Sub-second computation vs. minutes/hours for MIP
- **Implementation simplicity**: Easy to understand and modify vs. complex optimization
- **Robustness**: Handles data uncertainty and parameter changes gracefully
- **Scalability**: Efficient for large-scale problems (1000+ shipments)

### Pros / Cons vs Tier 1
**Pros:**
- Extremely fast computation (sub-second for any problem size)
- Simple implementation and intuitive logic
- Handles dynamic data and real-time requirements
- Easily customizable priority functions
- Robust to missing or uncertain data

**Cons:**
- No optimality guarantees (typically 95-99% of optimal)
- Performance depends on priority function design
- May miss complex interdependencies between shipments
- Limited ability to handle complex constraints
- Quality varies with parameter tuning

### When to use this Tier
- **Real-time operations** requiring immediate decisions
- **Large-scale problems** (500+ shipments) where MIP is infeasible
- **Dynamic environments** with frequent parameter changes
- **Resource-constrained systems** with limited computing power
- **Initial screening** before applying more sophisticated methods
- **Training and demonstration** due to transparent logic